In [ ]:
# Connect to Postgres datasets

import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.types import Date

# .env sits in the project root, one folder above notebooks/
load_dotenv('../.env')

# URL.create handles special characters in the password (@, :, /) safely,
# unlike building the connection string by hand with an f-string
url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=int(os.getenv("PG_PORT")),
    database=os.getenv("PG_DATABASE"),
)
engine = create_engine(url)

# test the connection
with engine.connect() as conn:
    print("Connected to database:", conn.execute(text("select current_database()")).scalar())

Connected to database: cafe_rewards


In [2]:
# Load all cleaned files
clean = '../data/cleaned/'

tables = {
    # table name in Postgres      : (csv file,                            columns to store as dates)
    'customers':                    ('customers_clean.csv',               ['became_member_on']),
    'customer_features':            ('customers_features.csv',            ['became_member_on']),
    'offers':                       ('offers_clean.csv',                  []),
    'events':                       ('events_clean.csv',                  []),
    'offer_funnel':                 ('offer_funnel.csv',                  []),
    'informational_offer_analysis': ('informational_offer_analysis.csv',  []),
}

loaded = {}
for table, (file, date_cols) in tables.items():
    df = pd.read_csv(clean + file, parse_dates=date_cols)     # parse_dates: text -> real dates

    # if_exists='replace' DROPS the table if it exists and recreates it
    df.to_sql(table, engine, schema='public', if_exists='replace', index=False,
              chunksize=10000, dtype={c: Date() for c in date_cols})

    loaded[table] = len(df)
    print(f"{table}: {len(df)} rows loaded")

customers: 14825 rows loaded
customer_features: 14825 rows loaded
offers: 10 rows loaded
events: 272388 rows loaded
offer_funnel: 66501 rows loaded
informational_offer_analysis: 13300 rows loaded
